In [1]:
# Imports

import os
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Definir rutas para no perderme

DATA_DIR = "../data"
RESULTS_DIR = "../results"
FIGURES_DIR = "../figures"

In [3]:
dataset_path = f"{DATA_DIR}/GSE125449"

import os
os.listdir(dataset_path)

['Set1', 'Set2']

In [4]:
# Leer set1

import os
import pandas as pd
import scanpy as sc
from scipy.io import mmread

set1_path = f"{DATA_DIR}/GSE125449/Set1"

# Leer matriz
X1 = mmread(os.path.join(set1_path, "matrix.mtx")).tocsr().T

# Leer barcodes
barcodes1 = pd.read_csv(
    os.path.join(set1_path, "barcodes.tsv"),
    header=None,
    sep="\t"
)[0].tolist()

# Leer features
features1 = pd.read_csv(
    os.path.join(set1_path, "features.tsv"),
    header=None,
    sep="\t"
)

# Crear AnnData
adata1 = sc.AnnData(
    X=X1,
    obs=pd.DataFrame(index=barcodes1),
    var=pd.DataFrame(index=features1[1].astype(str).tolist())
)

adata1.obs["batch"] = "Set1"

print(adata1)
print(adata1.shape)


AnnData object with n_obs × n_vars = 5115 × 20124
    obs: 'batch'
(5115, 20124)


/home/iruizdealda/.conda/envs/hcc_scanpy_clean/lib/python3.11/site-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [5]:
# Leer set2

set2_path = f"{DATA_DIR}/GSE125449/Set2"

# Leer matriz
X2 = mmread(os.path.join(set2_path, "matrix.mtx")).tocsr().T

# Leer barcodes
barcodes2 = pd.read_csv(
    os.path.join(set2_path, "barcodes.tsv"),
    header=None,
    sep="\t"
)[0].tolist()

# Leer features
features2 = pd.read_csv(
    os.path.join(set2_path, "features.tsv"),
    header=None,
    sep="\t"
)

# Crear AnnData
adata2 = sc.AnnData(
    X=X2,
    obs=pd.DataFrame(index=barcodes2),
    var=pd.DataFrame(index=features2[1].astype(str).tolist())
)

adata2.obs["batch"] = "Set2"

print(adata2)
print(adata2.shape)

AnnData object with n_obs × n_vars = 4831 × 19572
    obs: 'batch'
(4831, 19572)


/home/iruizdealda/.conda/envs/hcc_scanpy_clean/lib/python3.11/site-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Se cargaron por separado los subconjuntos Set1 y Set2 del dataset GSE125449.
Cada uno se transformó en un objeto AnnData, donde las filas representan células
y las columnas representan genes.
Set1 contiene 5115 células y 20124 genes, mientras que Set2 contiene
4831 células y 19572 genes.

Se añadió una columna llamada "batch" en la metadata de cada objeto AnnData
para identificar el origen de las células. De esta forma, las células de Set1
quedaron etiquetadas como "Set1" y las de Set2 como "Set2".
Esto permite conservar la información de procedencia tras la unión de ambos conjuntos.

In [6]:
# Hacer únicos los nombres de los genes
adata1.var_names_make_unique()
adata2.var_names_make_unique()

Durante la carga apareció un aviso indicando que algunos nombres de genes
estaban duplicados. Esto puede generar problemas en pasos posteriores,
como la búsqueda de genes concretos o la concatenación de datasets.
Por ello, se aplicó la función var_names_make_unique() para asegurar que
cada gen tuviera un identificador único dentro de cada objeto AnnData.

In [7]:
# Comprobar cuantos genes comparten
common_genes = adata1.var_names.intersection(adata2.var_names)
print("Genes comunes:", len(common_genes))

Genes comunes: 18372


Se calculó la intersección entre los nombres de genes de Set1 y Set2
para conocer cuántos genes estaban presentes en ambos subconjuntos.
Esta comprobación permite evaluar el grado de solapamiento entre las
variables antes de unir los dos objetos.

In [8]:
# Unir los datasets
adata = adata1.concatenate(adata2, join="outer")

# Comprobar el resultado

print(adata)
print(adata.shape)
adata.obs["batch"].value_counts()

/tmp/ipykernel_78668/2441378453.py:2: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata = adata1.concatenate(adata2, join="outer")


AnnData object with n_obs × n_vars = 9946 × 21324
    obs: 'batch'
(9946, 21324)


batch
0    5115
1    4831
Name: count, dtype: int64

Tras hacer únicos los nombres de genes, ambos subconjuntos se concatenaron
en un único objeto AnnData. Se utilizó join="outer" para conservar todos los
genes presentes en cualquiera de los dos sets. De esta forma, si un gen estaba
presente en uno de los subconjuntos pero no en el otro, se mantenía igualmente
en el objeto final, asignando valor 0 en las células donde no estaba disponible.

In [9]:
# Guardar datset

adata.write(f"{DATA_DIR}/adata_GSE125449_raw.h5ad")